# Gemma 4 E2B & E4B — Native Audio-Core + Cognitive-Core Atlas & 31B Token-258881 Bridge

- Scans `google/gemma-4-E2B-it` (`12` Conformer Audio layers + `35` Text layers) and `google/gemma-4-E4B-it` (`12` Conformer Audio layers + `42` Text layers).
- Solves exact `262,144`-token Procrustes/Ridge Audio Bridge onto Gemma 4 31B's `5,376`-D space (`audio_token_id = 258881`).
- Exports SHA-256 verified `safetensors` + Atlas JSON/NPZ to `/content/drive/MyDrive/gemma4_e2b_e4b_singular` and auto-disconnects (`runtime.unassign()`).

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA GPU required (T4 / L4 / A100)"
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GiB", flush=True)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_OUT = Path("/content/drive/MyDrive/gemma4_e2b_e4b_singular")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=5.0.0", "accelerate", "safetensors", "huggingface_hub", "tokenizers", "librosa", "soundfile", "scipy"],
    check=True,
)

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_e2b_e4b_audio_and_cognitive_singular_gpu.py"
res = subprocess.run([sys.executable, str(SCRIPT), "--selftest"], text=True, capture_output=True)
print(res.stdout, res.stderr, flush=True)
res.check_returncode()

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_e2b_e4b_audio_and_cognitive_singular_gpu.py"
DRIVE_OUT = Path("/content/drive/MyDrive/gemma4_e2b_e4b_singular")
LOG_PATH = Path("/content/e2b_e4b_singular.log")

env = os.environ.copy()
env["SINGULAR_DRIVE_DIR"] = str(DRIVE_OUT)
env["SINGULAR_OUT_DIR"] = "/content/e2b_e4b_singular_out"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONUNBUFFERED"] = "1"

with open(LOG_PATH, "w", encoding="utf-8") as lf:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        lf.write(line)
    rc = proc.wait()

shutil.copy2(LOG_PATH, DRIVE_OUT / "e2b_e4b_singular.log")
subprocess.run(["sync"], check=False)
assert rc == 0, f"E2B/E4B pipeline exited with {rc}"
print(subprocess.check_output(["sha256sum", "-c", "SHA256SUMS_E2B_E4B.txt"], cwd=str(DRIVE_OUT), text=True), flush=True)
print((DRIVE_OUT / "e2b_e4b_singular_report.json").read_text(encoding="utf-8"), flush=True)
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print(f"[Flush Note] {e}", flush=True)
runtime.unassign()